# Clase 5 · Tarea 02 — Mini proyecto: simulación financiera con NumPy

### Análisis cuantitativo de carteras de inversión

En este proyecto construirás, pieza por pieza, un simulador de carteras
financieras usando **exclusivamente NumPy**. Al final tendrás un reporte
completo de riesgo y rendimiento.

```
generar_retornos()
       │
       ▼
retorno_acumulado()
       │
       ├──▶ drawdown_maximo()
       │
       └──▶ sharpe_ratio()
                    │
                    ▼
             reporte_riesgo()   ← integra todo
```

**Instrucciones**

1. Implementa cada función reemplazando `raise NotImplementedError`.
2. **Resuélvelas en orden**: las partes 3-6 usan las anteriores.
3. **No cambies nombres ni parámetros**: las funciones se llaman entre sí.
4. Ejecuta los tests de cada parte hasta ver ✅.

> 🧠 Antes de programar cada función, dibuja el shape de entrada y salida.

| Parte | Función | Patrón NumPy |
|---|---|---|
| 1 | `generar_retornos` | `default_rng`, distribución normal |
| 2 | `retorno_acumulado` | `np.cumprod` |
| 3 | `drawdown_maximo` | `np.maximum.accumulate` |
| 4 | `sharpe_ratio` | reducción + escalar |
| 5 | `simular_carteras` | arrays 2-D, cumprod axis=1 |
| 6 | `reporte_riesgo` | integración |

In [ ]:
# Identifícate (esto ayuda si entregas el notebook).
NOMBRE = ""   # ✏️ escribe tu nombre completo
print("Tarea lista para resolver. Ejecuta cada bloque de tests tras implementar.")

## Ejercicio 1 · Generar retornos diarios

El primer paso de la simulación es generar `n` retornos diarios de un
activo financiero. Los retornos se modelan como distribución normal con
media `mu` (rendimiento diario esperado) y desviación `sigma` (volatilidad).

Implementa `generar_retornos(n, mu, sigma, seed)` que:
1. Use `np.random.default_rng(seed)` para reproducibilidad.
2. Genere `n` retornos con `rng.normal(loc=mu, scale=sigma, size=n)`.
3. Devuelva el array de retornos.

**Ejemplo:**
```python
r = generar_retornos(252, mu=0.001, sigma=0.02, seed=42)
# 252 retornos diarios (un año bursátil)
# media ≈ 0.001, std ≈ 0.02
```

In [ ]:
import numpy as np

def generar_retornos(n, mu=0.001, sigma=0.02, seed=42):
    rng = np.random.default_rng(seed)
    return rng.normal(loc=mu, scale=sigma, size=n)

In [ ]:
# === Tests visibles · Ejercicio 1 ===
import numpy as np
r = generar_retornos(252, mu=0.001, sigma=0.02, seed=42)
assert isinstance(r, np.ndarray)
assert len(r) == 252
assert abs(r.mean() - 0.001) < 0.005
assert abs(r.std() - 0.02) < 0.005
print("✅ Ejercicio 1: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 1 ===
import numpy as np
# Reproducibilidad: misma semilla, mismo resultado
r1 = generar_retornos(100, seed=0)
r2 = generar_retornos(100, seed=0)
assert np.array_equal(r1, r2)
# Semillas distintas dan resultados distintos
r3 = generar_retornos(100, seed=1)
assert not np.array_equal(r1, r3)
print("✅ Ejercicio 1: tests adicionales superados.")

## Ejercicio 2 · Retorno acumulado

Dado un array de retornos diarios, el **retorno acumulado** en el
día `t` es el producto de `(1 + r[i])` para `i` desde 0 hasta t.

Implementa `retorno_acumulado(retornos)` que:
1. Calcule el producto acumulado de `(1 + retornos)`.
2. Devuelva un array de la misma longitud donde cada elemento es
   el valor de la cartera (empezando desde 1.0).

Usa `np.cumprod`.

**Ejemplo:**
```python
r = np.array([0.05, -0.02, 0.03])
retorno_acumulado(r)
# → array([1.05, 1.029, 1.05987])
# dia 0: 1 * 1.05 = 1.05
# dia 1: 1.05 * 0.98 = 1.029
# dia 2: 1.029 * 1.03 = 1.05987
```

In [ ]:
import numpy as np

def retorno_acumulado(retornos):
    return np.cumprod(1 + retornos)

In [ ]:
# === Tests visibles · Ejercicio 2 ===
import numpy as np
r = np.array([0.05, -0.02, 0.03])
ac = retorno_acumulado(r)
assert len(ac) == 3
assert abs(ac[0] - 1.05) < 1e-9
assert abs(ac[1] - 1.05 * 0.98) < 1e-9
print("✅ Ejercicio 2: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 2 ===
import numpy as np
# Con retornos cero: la cartera no cambia
assert np.allclose(retorno_acumulado(np.zeros(5)), np.ones(5))
# Verificar que el ultimo valor es el producto total de (1+r)
_r = generar_retornos(252, seed=42)
_ac = retorno_acumulado(_r)
assert abs(_ac[-1] - np.prod(1 + _r)) < 1e-9
print("✅ Ejercicio 2: tests adicionales superados.")

## Ejercicio 3 · Drawdown máximo

El **drawdown** en un día `t` es la caída relativa desde el máximo
histórico hasta ese día:

```
drawdown[t] = (maximo_historico[t] - valor[t]) / maximo_historico[t]
```

El **drawdown máximo** es el mayor drawdown de toda la serie.

Implementa `drawdown_maximo(acumulado)` que:
1. Calcule el máximo histórico en cada día: `np.maximum.accumulate`.
2. Calcule el drawdown en cada día.
3. Devuelva el drawdown máximo (valor escalar).

**Ejemplo:**
```python
ac = np.array([1.0, 1.1, 0.9, 1.05, 0.85, 1.2])
drawdown_maximo(ac)  # → 0.2272... (caída de 1.1 a 0.85)
```

In [ ]:
import numpy as np

def drawdown_maximo(acumulado):
    maximo_hist = np.maximum.accumulate(acumulado)
    drawdowns = (maximo_hist - acumulado) / maximo_hist
    return float(drawdowns.max())

In [ ]:
# === Tests visibles · Ejercicio 3 ===
import numpy as np
ac = np.array([1.0, 1.1, 0.9, 1.05, 0.85, 1.2])
dd = drawdown_maximo(ac)
assert abs(dd - (1.1 - 0.85) / 1.1) < 1e-9
print("✅ Ejercicio 3: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 3 ===
import numpy as np
# Si la cartera solo sube, el drawdown es 0
subida = np.array([1.0, 1.1, 1.2, 1.3])
assert drawdown_maximo(subida) == 0.0
# Verificar usando retornos generados
_r = generar_retornos(252, seed=42)
_ac = retorno_acumulado(_r)
_dd = drawdown_maximo(_ac)
assert 0.0 <= _dd < 1.0
print("✅ Ejercicio 3: tests adicionales superados.")

## Ejercicio 4 · Sharpe Ratio aproximado

El **Sharpe Ratio** mide el rendimiento ajustado por riesgo:

```
sharpe = (media_retornos - tasa_libre_riesgo) / std_retornos
         * sqrt(252)   ← anualizar (252 días bursátiles)
```

Implementa `sharpe_ratio(retornos, tasa_libre=0.0)` que:
1. Calcule la media y desviación estándar del array de retornos.
2. Aplique la fórmula y anualice multiplicando por `np.sqrt(252)`.
3. Devuelva el Sharpe Ratio como float.

**Ejemplo:**
```python
r = np.array([0.01, 0.005, -0.003, 0.008])
sharpe_ratio(r)  # → algún valor > 0 si media > 0
```

In [ ]:
import numpy as np

def sharpe_ratio(retornos, tasa_libre=0.0):
    exceso = retornos.mean() - tasa_libre
    return float(exceso / retornos.std() * np.sqrt(252))

In [ ]:
# === Tests visibles · Ejercicio 4 ===
import numpy as np
r = np.array([0.01, 0.005, -0.003, 0.008])
sr = sharpe_ratio(r)
assert isinstance(sr, float)
# Con media positiva y tasa libre 0, el Sharpe debe ser positivo
assert sr > 0
print("✅ Ejercicio 4: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 4 ===
import numpy as np
# Retornos constantes: Sharpe indefinido (std=0), no lo testamos
# Retornos negativos: Sharpe < 0
_r_neg = np.full(100, -0.001)
_r_neg = _r_neg + np.random.default_rng(5).normal(0, 0.0001, 100)
# media negativa => sharpe negativo
assert sharpe_ratio(_r_neg) < 0
# Verificar formula manualmente
_r2 = generar_retornos(252, mu=0.001, sigma=0.02, seed=42)
_sr = sharpe_ratio(_r2)
_expected = float((_r2.mean() - 0.0) / _r2.std() * np.sqrt(252))
assert abs(_sr - _expected) < 1e-9
print("✅ Ejercicio 4: tests adicionales superados.")

## Ejercicio 5 · Simulación de múltiples carteras (Monte Carlo)

Implementa `simular_carteras(n_carteras, n_dias, mu, sigma, seed)` que:
1. Genere una matriz `(n_carteras, n_dias)` de retornos usando
   `rng.normal(mu, sigma, size=(n_carteras, n_dias))`.
2. Calcule el retorno final de cada cartera (último valor del
   retorno acumulado) usando `np.cumprod` con `axis=1`.
3. Devuelva un array de `n_carteras` retornos finales.

**Ejemplo:**
```python
finales = simular_carteras(1000, 252, 0.001, 0.02, seed=0)
# array de 1000 valores: retorno final de cada cartera simulada
```

Pista: `np.cumprod(1 + M, axis=1)[:, -1]` da el último valor de cada fila.

In [ ]:
import numpy as np

def simular_carteras(n_carteras, n_dias, mu=0.001, sigma=0.02, seed=42):
    rng = np.random.default_rng(seed)
    retornos = rng.normal(mu, sigma, size=(n_carteras, n_dias))
    acumulados = np.cumprod(1 + retornos, axis=1)
    return acumulados[:, -1]

In [ ]:
# === Tests visibles · Ejercicio 5 ===
import numpy as np
finales = simular_carteras(1000, 252, seed=0)
assert isinstance(finales, np.ndarray)
assert len(finales) == 1000
# Con mu=0.001 diario y 252 días, la media debe estar por encima de 1
assert finales.mean() > 1.0
print("✅ Ejercicio 5: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 5 ===
import numpy as np
# Reproducibilidad
f1 = simular_carteras(100, 50, seed=7)
f2 = simular_carteras(100, 50, seed=7)
assert np.array_equal(f1, f2)
# Cada retorno final es > 0 (no puede ser negativo, la cartera no puede valer < 0)
assert (finales > 0).all()
print("✅ Ejercicio 5: tests adicionales superados.")

## Ejercicio 6 · Análisis completo: reporte de riesgo

Integra las funciones anteriores en un reporte completo.

Implementa `reporte_riesgo(n_dias, mu, sigma, seed)` que:
1. Genere retornos con `generar_retornos`.
2. Calcule el retorno acumulado con `retorno_acumulado`.
3. Devuelva un diccionario con las siguientes métricas:
   - `'retorno_total'`: retorno final - 1 (ganancia porcentual).
   - `'drawdown_maximo'`: valor del drawdown máximo.
   - `'sharpe_ratio'`: Sharpe ratio anualizado.
   - `'volatilidad'`: desviación estándar de los retornos.

**Ejemplo:**
```python
r = reporte_riesgo(252, 0.001, 0.02, seed=42)
# {'retorno_total': 0.xx, 'drawdown_maximo': 0.xx,
#  'sharpe_ratio': 0.xx, 'volatilidad': 0.02x}
```

In [ ]:
import numpy as np

def reporte_riesgo(n_dias=252, mu=0.001, sigma=0.02, seed=42):
    retornos = generar_retornos(n_dias, mu, sigma, seed)
    acumulado = retorno_acumulado(retornos)
    return {
        'retorno_total':   float(acumulado[-1] - 1),
        'drawdown_maximo': drawdown_maximo(acumulado),
        'sharpe_ratio':    sharpe_ratio(retornos),
        'volatilidad':     float(retornos.std()),
    }

In [ ]:
# === Tests visibles · Ejercicio 6 ===
import numpy as np
rep = reporte_riesgo(252, 0.001, 0.02, seed=42)
assert set(rep.keys()) == {'retorno_total', 'drawdown_maximo', 'sharpe_ratio', 'volatilidad'}
assert isinstance(rep['retorno_total'], float)
assert 0.0 <= rep['drawdown_maximo'] < 1.0
print("✅ Ejercicio 6: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 6 ===
import numpy as np
# Verificar que las metricas son consistentes con las funciones individuales
_r = generar_retornos(252, 0.001, 0.02, seed=42)
_ac = retorno_acumulado(_r)
_rep = reporte_riesgo(252, 0.001, 0.02, seed=42)
assert abs(_rep['retorno_total'] - float(_ac[-1] - 1)) < 1e-9
assert abs(_rep['drawdown_maximo'] - drawdown_maximo(_ac)) < 1e-9
assert abs(_rep['sharpe_ratio'] - sharpe_ratio(_r)) < 1e-9
assert abs(_rep['volatilidad'] - float(_r.std())) < 1e-9
print("✅ Ejercicio 6: tests adicionales superados.")

---
## ¡Proyecto terminado!

Si todos los tests pasan, construiste un simulador financiero funcional.

### Lo que aprendiste con este proyecto

- **`np.cumprod`** para acumular multiplicaciones (retornos compuestos).
- **`np.maximum.accumulate`** para calcular el máximo histórico de una serie.
- **Arrays 2-D** para simular múltiples escenarios en paralelo (sin bucles).
- **Composición de funciones** NumPy para construir análisis complejos.

### Reto opcional (sin calificar)

1. Usa `simular_carteras` para generar 10.000 carteras durante 252 días
   y calcula:
   - ¿Qué porcentaje termina con ganancia?
   - ¿Cuál es el percentil 5 del retorno final (VaR al 95%)?
   - Grafica el histograma de retornos finales con `matplotlib`.

2. ¿Cómo cambiaría el Sharpe Ratio si la tasa libre de riesgo fuera 0.04%
   diario (≈10% anual)? Modifica `sharpe_ratio` para incluirla.

> 💡 En finanzas cuantitativas, este tipo de simulación se usa para
> calcular el **Value at Risk (VaR)** y el **Expected Shortfall (ES)**,
> que son las métricas estándar de gestión de riesgo.